# YELLOW RAW DATA ANALYSIS

In [ ]:
import findspark
findspark.init()
import pyspark  
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampNTZType,
    IntegerType, DoubleType, LongType
)
from pyspark.sql import functions as f
from pathlib import Path
import pandas as pd

In [ ]:
projectRoot = Path.cwd().parents[2]

ZONE_CSV = Path(projectRoot)/ "data" / "raw" / "tlc_trip_record" / "taxi_zone_lookup.csv"
PATH_PARQUETS = Path(projectRoot) / "data" / "raw" / "tlc_trip_record" / "yellow"
OUTPUT = Path(projectRoot) / "data" / "clean" / "clean_tlc_trip_record" / "yellow"

MIN_DATETIME = "2020-01-01"
MAX_DATETIME =  "2026-01-01"
MIN_YEAR = 2020
MAX_YEAR =  2025

MAX_TRIP_MILES = 50 
MIN_TRIP_MILES = 0.1
MAX_TRIP_TIME = 10_800
MIN_TRIP_TIME = 60

MIN_FARE = 1.0
MAX_FARE = 500.00

TIP_PCT = 0.25

UNKNOWN_LOCATIONS = [264, 265]
CBD_CORE_ZONES = [161, 162, 163, 164, 186, 229, 230, 231]
AIRPORT_ZONES = [1, 132, 138]

In [ ]:
projectRoot

# Init Spark

In [ ]:
def stop(spark):
    spark.sparkContext.stop()

spark = (SparkSession.builder
        .master("local[6]")  
        .config("spark.driver.memory", "8g")
        .config("spark.executor.memory", "8g")
        .config("spark.sql.shuffle.partitions", "100")
        .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
        .config("spark.driver.maxResultSize", "1g")
        .config("spark.sql.session.timeZone", "America/New_York")
        .config("spark.sql.timestampType", "TIMESTAMP_NTZ")
        # Network binding fixes for macOS
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .getOrCreate())

In [ ]:
spark.sparkContext

# Raw Data

**Yellow Taxi Trip Data - Variable Descriptions**

| Variable Name | Python Type | Description |
| :--- | :--- | :--- |
| **VendorID** | `int` | Code for the taxi technology provider. |
| **tpep_pickup_datetime** | `datetime` | Date and time when the taxi meter was turned on. |
| **tpep_dropoff_datetime** | `datetime` | Date and time when the taxi meter was turned off. |
| **passenger_count** | `int` | Number of passengers in the taxi. |
| **trip_distance** | `float` | Trip distance in miles as reported by the taximeter. |
| **RatecodeID** | `int` | Final rate code applied at the end of the trip (e.g., standard, airport). |
| **store_and_fwd_flag** | `str` | Indicates if trip data was stored in vehicle memory before transmission ('Y' or 'N'). |
| **PULocationID** | `int` | Taxi Zone ID where the passenger was picked up. |
| **DOLocationID** | `int` | Taxi Zone ID where the passenger was dropped off. |
| **payment_type** | `int` | Code for the payment method used (e.g., credit card, cash). |
| **fare_amount** | `float` | Time-and-distance fare calculated by the meter. |
| **extra** | `float` | Miscellaneous extras and surcharges. |
| **mta_tax** | `float` | Automatically triggered tax based on the metered rate. |
| **tip_amount** | `float` | Tip amount (credit card tips only; cash tips not included). |
| **tolls_amount** | `float` | Total amount of all tolls paid during the trip. |
| **improvement_surcharge** | `float` | Surcharge assessed at the start of the trip. |
| **total_amount** | `float` | Total amount charged to the passenger (excluding cash tips). |
| **congestion_surcharge** | `float` | Amount collected for NYS congestion surcharge. |
| **airport_fee** | `float` | Fee for pickups at LaGuardia or JFK airports. |
| **cbd_congestion_fee** | `float` | Per-trip charge for the MTA's Congestion Relief Zone (effective Jan 5, 2025). |

Prior to analysis, we should do some kind of features selection, we need to identify the most relevant variables. This step is crucial since it reduces dimensionality of the dataset.

### **Selected Features - NYC Yellow Taxi Trip Records**

#### Trip Timestamps
- **`tpep_pickup_datetime`**  
  The exact date and time when the trip started (meter engaged).

- **`tpep_dropoff_datetime`**  
  The exact date and time when the trip ended (meter disengaged).

#### Trip Route & Distance
- **`trip_distance`**  
  The total distance traveled during the trip, measured in miles.

- **`PULocationID`**  
  The Taxi Zone ID where the passenger was picked up.

- **`DOLocationID`**  
  The Taxi Zone ID where the passenger was dropped off.

#### Fare & Charges
- **`tip_amount`**  
  The amount of tip given by the passenger (credit card tips only).

- **`total_amount`**  
  The final total amount charged to the passenger.

- **`congestion_surcharge`**  
  Additional charge for trips within New York State congestion zones.

- **`airport_fee`**  
  A fee applied for pickups at LaGuardia or John F. Kennedy airports.

## Load Data

In [ ]:
def load_yellow(spark: SparkSession) -> DataFrame:
    """ 
        Load parquet 
    """
    schema = StructType([
        StructField("tpep_pickup_datetime",     TimestampNTZType()),
        StructField("tpep_dropoff_datetime",    TimestampNTZType()),
        StructField("trip_distance",            DoubleType()),
        StructField("PULocationID",             LongType()),
        StructField("DOLocationID",             LongType()),
        StructField("tip_amount",               DoubleType()),
        StructField("total_amount",             DoubleType()),
        StructField("congestion_surcharge",     DoubleType()),
        StructField("airport_fee",              DoubleType()),
        StructField("cbd_congestion_fee",       DoubleType())
    ])

    path = str(PATH_PARQUETS / "*.parquet")
    df = spark.read.schema(schema).parquet(path)

    df = df.withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')
    df = df.withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime')
    return df

def load_zones(spark: SparkSession) -> DataFrame:
    path = str(ZONE_CSV)
    df = spark.read.option("header", "true").csv(path)
    df = df.select(
        f.col("LocationID").cast(IntegerType()).alias("LocationID"),
        f.col("Borough").alias("Borough"),
        f.col("Zone").alias("Zone"),
        f.col("service_zone").alias("service_zone"),
    )
    return df

In [ ]:
raw_data = load_yellow(spark)

Although there are warnings that seem to be related to the path of the raw data, it loads all the data without any problem. Suppose we can just ignore them.

In [ ]:
zones = load_zones(spark)

## Basic Analysis: Stats, Nulls, Outliers

### Zones

In [ ]:
zones.printSchema()

In [ ]:
zones.show(3)

### Trips

In [ ]:
raw_data.show(5)
total_rows = raw_data.count()
print(f"Shape: {total_rows} rows, {len(raw_data.columns)} columns")

In [ ]:
raw_data.printSchema()

In [ ]:
num_cols = ["trip_distance", "tip_amount", "total_amount",
            "congestion_surcharge", "airport_fee","cbd_congestion_fee"]


#### Summary

In [ ]:
raw_summary = raw_data.select(num_cols).summary().toPandas()

In [ ]:
raw_summary

### Nulls

In [ ]:
def count_nulls(df:DataFrame):
    null_counts = {}
    for col in df.columns:
        nulls = df.where(f.col(col).isNull()).count()
        null_counts[col] = nulls

    for col, n in null_counts.items():
        percent = (n / total_rows * 100) if total_rows > 0 else 0
        print(f"{col}: {n:,} ({percent:.2f}%)")

In [ ]:
count_nulls(raw_data)

- congestion_surcharge: 19,473,965 (8.89%)
- airport_fee: 47,475,792 (21.67%)


- `congestion_surcharge`: Input 2.5 if the PULocationID or DOLocationID is in CBD_CORE_ZONES, otherwise, replace null with zero.

- `airport_fee`: Two conditions
    
    1. Delete the observations where any locationID (PU or DO) is from an airport [1, 132, 138]
    
    2. If the locationID is not an airport replace it with zero.
- `cbd_congestion_fee` : Is showhing a big number of nulls. This is because this fee was introduced in `2025`, pre 2025 will be nulls. If there are nulls in 2025 observations we can replace it with zero.

In [ ]:
aiport_cond_one = raw_data.where(
    ((f.col("DOLocationID").isin(AIRPORT_ZONES)) | (f.col("PULocationID").isin(AIRPORT_ZONES))) 
    & (f.col("airport_fee").isNull())
).count()

aiport_cond_two = raw_data.where(
    ~((f.col("DOLocationID").isin(AIRPORT_ZONES)) | (f.col("PULocationID").isin(AIRPORT_ZONES))) 
    & (f.col("airport_fee").isNull())
).count()

Number of obeservations that satisfy condition 1, these will be deleted

In [ ]:
print(aiport_cond_one)

percent = (aiport_cond_one / total_rows * 100) if total_rows > 0 else 0
print(f"To be deleted: {aiport_cond_one:,} ({percent:.2f}%)")

Number of obeservations that satisfy condition 2, these will be filled w/ 0

In [ ]:
print(aiport_cond_two)

percent = (aiport_cond_two / total_rows * 100) if total_rows > 0 else 0
print(f"Filled w/ 0: {aiport_cond_two:,} ({percent:.2f}%)")

`LocationID`: Decided to delete the rows where any locationID is 264 or 265 which are either "unknown" or "N/A" (Outside of NYC)

### Negative numbers

In [ ]:
with_negatives = ['trip_distance', 'tip_amount', 'total_amount', 
                  'congestion_surcharge', 'airport_fee', 'cbd_congestion_fee']

def count_negatives(df:DataFrame, columns:list):
    neg_counts = {}
    for col in columns:
        if col in df.columns:
            negs = df.where(f.col(col) < 0).count()
            neg_counts[col] = negs

    for col, n in neg_counts.items():
        percent = (n / total_rows * 100) if total_rows > 0 else 0
        print(f"{col}: {n:,} ({percent:.2f}%)")

In [ ]:
count_negatives(raw_data, with_negatives)

Shouldn't be problematic if we just filter the data.

## Ranges

### Tips

In [ ]:
correct_tips_wo_negs = (
    raw_data
    .where(
        f.col('tip_amount') > 0
    )
)

correct_tips = (
    correct_tips_wo_negs
    .where(
        (f.col('tip_amount') <= f.col('total_amount') * 0.2)
    )
)

print(f"Without negatives data: {correct_tips_wo_negs.count()}")
print(f"With max tip value: {correct_tips.count()}")
print(f"Percentage: {((correct_tips_wo_negs.count() - correct_tips.count()) / correct_tips_wo_negs.count())*100}%")

### General

In [ ]:
def percentiles(df, column):
    df.select(
        f.percentile_approx(column, [0.95, 0.99, 0.999, 0.9999])
        .alias(f"percentiles {column}")
    ).show(truncate=False)

In [ ]:
problematic_cols = ["trip_distance", "tip_amount", "total_amount", 'trip_duration']

df_with_duration = (
    raw_data
    .withColumn(
        'trip_duration',
        f.unix_timestamp('dropoff_datetime') - f.unix_timestamp('pickup_datetime')
    )
)

In [ ]:
for c in problematic_cols:
    df_no_negs = df_with_duration.filter(f.col(c) > 0)
    percentiles(df_no_negs, c)

# +-------------------------------+
# |percentiles trip_distance      |
# +-------------------------------+
# |[13.4, 19.92, 29.56, 398608.62]|
# +-------------------------------+

# +------------------------------------+
# |percentiles tip_amount              |
# +------------------------------------+
# |[11.84, 17.47, 30.0, 1.3339136353E8]|
# +------------------------------------+

                                                                                
# +--------------------------------+
# |percentiles total_amount        |
# +--------------------------------+
# |[70.94, 99.78, 162.3, 1000003.8]|
# +--------------------------------+

# +------------------------------+
# |percentiles trip_duration     |
# +------------------------------+
# |[2465, 4013, 14515, 619323311]|
# +------------------------------+


# FINAL DATA

The dataset obtained after running `src/transformations/yellowTransformations.py`

In [ ]:
final_data = spark.read.parquet(str(OUTPUT))

In [ ]:
final_data.printSchema()

In [ ]:
final_data_rows = final_data.count()
total_rows - final_data_rows 

In [ ]:
print(f'Erased PCT:{((total_rows-final_data_rows)/total_rows*100):.2f}%')

In [ ]:
count_nulls(final_data)

In [ ]:
count_negatives(final_data, with_negatives)

In [ ]:
num_cols = ['trip_distance', 'tip_amount', 'total_amount', 
                  'congestion_surcharge', 'airport_fee', 'trip_duration']

final_summary = final_data.select(num_cols).summary().toPandas()

In [ ]:
final_summary

In [ ]:
stop(spark)